# Tutorial 3: Time Series Analysis

Welcome to the third tutorial in our statistical learning series! In this notebook, we'll explore time series analysis - a field dedicated to analyzing data points collected or indexed in time order to extract meaningful patterns and predictions.

## Learning Objectives

By the end of this tutorial, you'll be able to:
- Decompose time series into trend, seasonal, and residual components
- Compute rolling statistics and detect change points
- Identify and test for trends using robust statistical methods
- Build forecasting models with ARIMA and exponential smoothing
- Evaluate forecast accuracy and interpret results

## 1. Setup and Introduction

Let's begin by importing the necessary libraries.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import our time series utilities
from statistics_lessons.trend_features.trend_features import (
    rolling_mean,
    rolling_std,
    trend_features,
    detect_trend
)
from statistics_lessons.trend_features.seasonal import decompose_series
from statistics_lessons.trend_features.forecasting import (
    arima_forecast,
    exponential_smoothing_forecast
)
from statistics_lessons.projects.data_loaders import load_sunspots

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Understanding Time Series Data

Time series data is a sequence of data points, typically measured at successive time intervals. Let's start by generating a simple synthetic time series to understand the core concepts.

In [ ]:
# Generate a synthetic time series with trend, seasonality, and noise
np.random.seed(42)

# Create a date range
dates = pd.date_range(start='2020-01-01', periods=365, freq='D')

# Generate components
t = np.arange(len(dates))
trend = 0.1 * t  # Linear trend
seasonality = 10 * np.sin(2 * np.pi * t / 30)  # Monthly seasonality
noise = np.random.normal(0, 2, len(dates))  # Random noise

# Combine components
y = trend + seasonality + noise

# Create a time series
ts = pd.Series(y, index=dates)

# Plot the time series
plt.figure(figsize=(12, 6))
plt.plot(ts)
plt.title('Synthetic Time Series')
plt.xlabel('Date')
plt.ylabel('Value')
plt.show()

### 2.1 Time Series Components

Time series data typically consists of several components:

1. **Trend**: The long-term progression of the series (increasing, decreasing, or stable)
2. **Seasonality**: Regular and predictable patterns that repeat over fixed intervals
3. **Cyclical patterns**: Fluctuations that aren't of fixed calendar frequency
4. **Residual/Random**: The irregular component that remains after the other components are removed

Let's decompose our synthetic time series to identify these components.

In [ ]:
# Decompose the time series
period = 30  # We know it has monthly seasonality (30 days)
decomposition = decompose_series(ts, period=period)

# Plot the decomposition
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

axes[0].plot(ts)
axes[0].set_title('Original Time Series')

axes[1].plot(decomposition['trend'])
axes[1].set_title('Trend Component')

axes[2].plot(decomposition['seasonal'])
axes[2].set_title('Seasonal Component')

axes[3].plot(decomposition['resid'])
axes[3].set_title('Residual Component')

plt.tight_layout()
plt.show()

### Interactive Exercise 1: Analyzing Decomposition Components

🔍 **Analyze the decomposition plot and answer these questions:**

1. How does the extracted trend compare to the true trend we generated (0.1 * t)?
2. Does the seasonal component accurately capture the monthly pattern we defined?
3. What characteristics do you notice in the residual component?
4. What happens if we decompose the series with an incorrect period (e.g., 7 days instead of 30)?

<details>
<summary>Click for answers</summary>

1. The extracted trend should be approximately linear with a positive slope close to 0.1, though it might be slightly different due to the decomposition algorithm and the influence of noise.

2. The seasonal component should nicely capture the sinusoidal pattern with a period of 30 days that we defined in our synthetic data.

3. The residual component should look like random noise without any obvious pattern, which is what we expect since we added Gaussian noise to our synthetic data.

4. If we decompose with an incorrect period:
   - The seasonal component would try to fit a pattern that doesn't match the true seasonality
   - Some of the true seasonal pattern would leak into the residual component
   - The trend might also be affected, as the decomposition algorithm attempts to balance the components
</details>

Let's try decomposing with an incorrect period to see what happens:

In [ ]:
# Decompose with incorrect period
wrong_period = 7  # Weekly instead of monthly
wrong_decomposition = decompose_series(ts, period=wrong_period)

# Plot the decomposition
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

axes[0].plot(ts)
axes[0].set_title('Original Time Series')

axes[1].plot(wrong_decomposition['trend'])
axes[1].set_title('Trend Component (Wrong Period)')

axes[2].plot(wrong_decomposition['seasonal'])
axes[2].set_title('Seasonal Component (Wrong Period)')

axes[3].plot(wrong_decomposition['resid'])
axes[3].set_title('Residual Component (Wrong Period)')

plt.tight_layout()
plt.show()

## 3. Rolling Statistics and Moving Averages

Rolling statistics are useful for analyzing how time series properties change over time. Let's explore rolling means and standard deviations.

In [ ]:
# Calculate rolling statistics
window = 30  # 30-day window

roll_mean = rolling_mean(ts, window)
roll_std = rolling_std(ts, window)

# Plot the original series with rolling statistics
plt.figure(figsize=(12, 6))
plt.plot(ts, alpha=0.5, label='Original')
plt.plot(roll_mean, label=f'{window}-day Rolling Mean')
plt.plot(roll_std, label=f'{window}-day Rolling Std')
plt.legend()
plt.title('Time Series with Rolling Statistics')
plt.show()

### 3.1 Detecting Change Points

Change points are abrupt variations in the statistical properties of a time series. Let's introduce a change point to our synthetic data and detect it using rolling statistics.

In [ ]:
# Create a series with a change point
dates_extended = pd.date_range(start='2020-01-01', periods=730, freq='D')  # 2 years
t_extended = np.arange(len(dates_extended))

# First year - same as before
trend1 = 0.1 * t_extended[:365]
seasonality1 = 10 * np.sin(2 * np.pi * t_extended[:365] / 30)
noise1 = np.random.normal(0, 2, 365)
y1 = trend1 + seasonality1 + noise1

# Second year - increased trend and variance
trend2 = 0.1 * t_extended[365:] + 15  # Level shift
seasonality2 = 15 * np.sin(2 * np.pi * t_extended[365:] / 30)  # Amplitude change
noise2 = np.random.normal(0, 4, 365)  # Increased variance
y2 = trend2 + seasonality2 + noise2

# Combine the two periods
y_combined = np.concatenate([y1, y2])
ts_change = pd.Series(y_combined, index=dates_extended)

# Calculate rolling statistics
roll_mean_change = rolling_mean(ts_change, window)
roll_std_change = rolling_std(ts_change, window)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(ts_change, alpha=0.5, label='Series with Change Point')
plt.plot(roll_mean_change, label=f'{window}-day Rolling Mean')
plt.plot(roll_std_change, label=f'{window}-day Rolling Std')
plt.axvline(x=dates_extended[365], color='r', linestyle='--', label='Change Point')
plt.legend()
plt.title('Detecting Change Points with Rolling Statistics')
plt.show()

## 4. Trend Detection and Analysis

Detecting and quantifying trends in time series data is crucial for understanding long-term patterns. Let's use the `trend_features` utility to compute robust trend statistics.

In [ ]:
# Use trend_features to compute Theil-Sen slope and Mann-Kendall p-value
window = 90  # 90-day window for trend analysis
trends = trend_features(ts, window)

# Detect significant trends (p < 0.05)
significant_trends = detect_trend(ts, window, alpha=0.05)

# Plot the time series with trend information
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Original series
axes[0].plot(ts)
axes[0].set_title('Original Time Series')

# Theil-Sen slope
axes[1].plot(trends['theil_sen_slope'])
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_title('Theil-Sen Slope (Trend Magnitude)')

# Significant trends
axes[2].plot(significant_trends)
axes[2].set_title('Significant Trends (Mann-Kendall p < 0.05)')

plt.tight_layout()
plt.show()

# Print some information about the detected trends
trend_count = significant_trends.sum()
trend_percentage = 100 * trend_count / len(significant_trends.dropna())
print(f"Significant trends detected in {trend_percentage:.2f}% of the windows")
print(f"Average Theil-Sen slope: {trends['theil_sen_slope'].mean():.4f}")

### 4.1 Understanding Theil-Sen and Mann-Kendall Tests

The Theil-Sen estimator and Mann-Kendall test are robust non-parametric methods for trend analysis:

1. **Theil-Sen Estimator**: Computes the median of all pairwise slopes in the data, making it resistant to outliers
2. **Mann-Kendall Test**: Tests for monotonic trends without assuming linearity or normality

Let's apply these methods to a series with a clearer trend:

In [ ]:
# Create a series with different trend patterns
dates_trend = pd.date_range(start='2020-01-01', periods=730, freq='D')
t_trend = np.arange(len(dates_trend))

# First segment: No trend
segment1 = np.random.normal(10, 2, 180)

# Second segment: Positive trend
segment2_x = np.arange(180)
segment2 = 0.05 * segment2_x + np.random.normal(10, 2, 180)

# Third segment: No trend
segment3 = np.random.normal(20, 2, 180)

# Fourth segment: Negative trend
segment4_x = np.arange(180)
segment4 = 25 - 0.07 * segment4_x + np.random.normal(10, 2, 180)

# Combine segments
y_segments = np.concatenate([segment1, segment2, segment3, segment4])
ts_segments = pd.Series(y_segments, index=dates_trend[:len(y_segments)])

# Apply trend detection with a 60-day window
window_segments = 60
trends_segments = trend_features(ts_segments, window_segments)
significant_trends_segments = detect_trend(ts_segments, window_segments, alpha=0.05)

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Original series with segment boundaries
axes[0].plot(ts_segments)
axes[0].axvline(x=dates_trend[180], color='r', linestyle='--')
axes[0].axvline(x=dates_trend[360], color='r', linestyle='--')
axes[0].axvline(x=dates_trend[540], color='r', linestyle='--')
axes[0].set_title('Time Series with Different Trend Patterns')

# Theil-Sen slope
axes[1].plot(trends_segments['theil_sen_slope'])
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].axvline(x=dates_trend[180], color='r', linestyle='--')
axes[1].axvline(x=dates_trend[360], color='r', linestyle='--')
axes[1].axvline(x=dates_trend[540], color='r', linestyle='--')
axes[1].set_title('Theil-Sen Slope')

# Significant trends
axes[2].plot(significant_trends_segments)
axes[2].axvline(x=dates_trend[180], color='r', linestyle='--')
axes[2].axvline(x=dates_trend[360], color='r', linestyle='--')
axes[2].axvline(x=dates_trend[540], color='r', linestyle='--')
axes[2].set_title('Significant Trends (Mann-Kendall p < 0.05)')

plt.tight_layout()
plt.show()

### Interactive Exercise 2: Analyzing Trend Patterns

🔍 **Analyze the trend patterns in the segmented time series:**

1. In which segments does the Theil-Sen slope indicate a significant trend?
2. How well does the Mann-Kendall test detect the transitions between different trend patterns?
3. What happens to the trend metrics near the boundaries between segments?
4. How might you use these tools to detect trend changes in real-world data?

<details>
<summary>Click for answers</summary>

1. The Theil-Sen slope should show positive values in the second segment (181-360) and negative values in the fourth segment (541-720). The first and third segments should have slopes close to zero.

2. The Mann-Kendall test should identify significant trends in the second and fourth segments, but not in the first and third segments. There may be a delay in detecting the transitions due to the window-based approach.

3. Near boundaries between segments:
   - There's typically a lag in detecting new trends since the window contains data from both segments
   - The slope gradually transitions from one pattern to another
   - Statistical significance may be reduced during transitions

4. Applications to real-world data:
   - Detect structural changes in economic indicators
   - Identify shifts in climate data or environmental monitoring
   - Monitor changes in business metrics like sales or customer engagement
   - The window size should be chosen based on the expected duration of trends and noise level
</details>

## 5. Working with Real Data: Sunspots Analysis

Let's apply our time series techniques to a real dataset: the sunspot series, which records the number of sunspots observed each year since 1700.

In [ ]:
# Load the sunspots dataset
sunspots = load_sunspots()

# Basic information about the series
print(f"Sunspots data: {len(sunspots)} observations from {sunspots.index.min()} to {sunspots.index.max()}")
print(f"Mean: {sunspots.mean():.2f}, Min: {sunspots.min()}, Max: {sunspots.max()}")

# Plot the series
plt.figure(figsize=(12, 6))
plt.plot(sunspots)
plt.title('Annual Sunspot Counts (1700-2008)')
plt.xlabel('Year')
plt.ylabel('Sunspot Count')
plt.grid(True)
plt.show()

### 5.1 Decomposing the Sunspot Series

The sunspot series shows a clear cyclical pattern known as the "solar cycle," which lasts approximately 11 years. Let's decompose this series.

In [ ]:
# Decompose the sunspot series
# The solar cycle is approximately 11 years
decomposed_sunspots = decompose_series(sunspots, period=11)

# Plot the decomposition
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(sunspots)
axes[0].set_title('Original Sunspot Series')

axes[1].plot(decomposed_sunspots['trend'])
axes[1].set_title('Trend Component')

axes[2].plot(decomposed_sunspots['seasonal'])
axes[2].set_title('Cyclic Component (11-year solar cycle)')

plt.tight_layout()
plt.show()

### 5.2 Analyzing Long-term Trends in Sunspot Activity

Let's examine long-term trends in sunspot activity using our trend detection utilities.

In [ ]:
# Compute trends with a 22-year window (two solar cycles)
window_sunspots = 22
trends_sunspots = trend_features(sunspots, window_sunspots)
significant_trends_sunspots = detect_trend(sunspots, window_sunspots, alpha=0.05)

# Plot trends
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(sunspots)
axes[0].set_title('Sunspot Series')

axes[1].plot(trends_sunspots['theil_sen_slope'])
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_title(f'Theil-Sen Slope ({window_sunspots}-year window)')

axes[2].plot(significant_trends_sunspots)
axes[2].set_title('Significant Trends (Mann-Kendall p < 0.05)')

plt.tight_layout()
plt.show()

# Calculate proportion of periods with significant trends
trend_count = significant_trends_sunspots.sum()
trend_percentage = 100 * trend_count / len(significant_trends_sunspots.dropna())
print(f"Significant trends detected in {trend_percentage:.2f}% of the windows")

## 6. Forecasting Time Series

Now, let's move on to forecasting future values in a time series. We'll explore two common approaches:

1. **ARIMA (AutoRegressive Integrated Moving Average)**: A flexible model that captures autoregressive, differencing, and moving average components
2. **Exponential Smoothing**: A method that gives exponentially decreasing weights to past observations

### 6.1 ARIMA Forecasting

In [ ]:
# Split the sunspot series into training and test sets
split_year = 1990
train_sunspots = sunspots[sunspots.index < split_year]
test_sunspots = sunspots[sunspots.index >= split_year]

# Define ARIMA order (p, d, q)
# p: Autoregressive lags
# d: Differencing
# q: Moving average lags
order = (5, 1, 0)  # AR(5) with first differencing

# Forecast
forecast_steps = len(test_sunspots)
arima_predictions = arima_forecast(train_sunspots, order, forecast_steps)

# Plot the results
plt.figure(figsize=(12, 6))
plt.plot(train_sunspots, label='Training Data')
plt.plot(test_sunspots, label='Test Data')
plt.plot(test_sunspots.index, arima_predictions, 'r--', label=f'ARIMA{order} Forecast')
plt.title('ARIMA Forecast of Sunspot Activity')
plt.xlabel('Year')
plt.ylabel('Sunspot Count')
plt.legend()
plt.grid(True)
plt.show()

# Calculate forecast accuracy
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt

rmse_arima = sqrt(mean_squared_error(test_sunspots, arima_predictions))
mae_arima = mean_absolute_error(test_sunspots, arima_predictions)

print(f"ARIMA Forecast Accuracy:")
print(f"RMSE: {rmse_arima:.2f}")
print(f"MAE: {mae_arima:.2f}")
print(f"MAPE: {100 * np.mean(np.abs((test_sunspots - arima_predictions) / test_sunspots)):.2f}%")

### 6.2 Exponential Smoothing

In [ ]:
# Apply exponential smoothing with trend and seasonality
exp_smooth_predictions = exponential_smoothing_forecast(
    train_sunspots,
    trend='add',
    seasonal='add',
    seasonal_periods=11,  # 11-year solar cycle
    steps=forecast_steps
)

# Plot the results
plt.figure(figsize=(12, 6))
plt.plot(train_sunspots, label='Training Data')
plt.plot(test_sunspots, label='Test Data')
plt.plot(test_sunspots.index, exp_smooth_predictions, 'g--', label='Exponential Smoothing Forecast')
plt.title('Exponential Smoothing Forecast of Sunspot Activity')
plt.xlabel('Year')
plt.ylabel('Sunspot Count')
plt.legend()
plt.grid(True)
plt.show()

# Calculate forecast accuracy
rmse_exp = sqrt(mean_squared_error(test_sunspots, exp_smooth_predictions))
mae_exp = mean_absolute_error(test_sunspots, exp_smooth_predictions)

print(f"Exponential Smoothing Forecast Accuracy:")
print(f"RMSE: {rmse_exp:.2f}")
print(f"MAE: {mae_exp:.2f}")
print(f"MAPE: {100 * np.mean(np.abs((test_sunspots - exp_smooth_predictions) / test_sunspots)):.2f}%")

### 6.3 Comparing Forecasting Methods

In [ ]:
# Compare ARIMA and Exponential Smoothing
plt.figure(figsize=(12, 6))
plt.plot(train_sunspots, label='Training Data')
plt.plot(test_sunspots, label='Test Data')
plt.plot(test_sunspots.index, arima_predictions, 'r--', label=f'ARIMA{order}')
plt.plot(test_sunspots.index, exp_smooth_predictions, 'g--', label='Exponential Smoothing')
plt.title('Comparison of Forecasting Methods')
plt.xlabel('Year')
plt.ylabel('Sunspot Count')
plt.legend()
plt.grid(True)
plt.show()

# Summarize comparison
print("Forecast Method Comparison:")
print(f"{'Method':<20} {'RMSE':<10} {'MAE':<10} {'MAPE':<10}")
print("-" * 50)
print(f"{'ARIMA':<20} {rmse_arima:<10.2f} {mae_arima:<10.2f} {100 * np.mean(np.abs((test_sunspots - arima_predictions) / test_sunspots)):<10.2f}%")
print(f"{'Exponential Smoothing':<20} {rmse_exp:<10.2f} {mae_exp:<10.2f} {100 * np.mean(np.abs((test_sunspots - exp_smooth_predictions) / test_sunspots)):<10.2f}%")

### Interactive Exercise 3: Forecasting Analysis

🔍 **Analyze the forecasting results and consider these questions:**

1. Which forecasting method performed better for the sunspot series? Why might that be?
2. How well do both methods capture the cyclical nature of sunspot activity?
3. How might you improve the forecasts? Consider model parameters and alternative approaches.
4. What are the limitations of these forecasting methods for this type of data?

<details>
<summary>Click for answers</summary>

1. The better-performing method depends on the metrics. Usually, one method has lower error metrics than the other:
   - ARIMA may perform better if the data has strong autoregressive components
   - Exponential smoothing might perform better if the seasonal pattern is very regular

2. Both methods attempt to capture the cyclical pattern, but their success depends on:
   - For ARIMA: Whether the order parameters are appropriate for the cycle length
   - For exponential smoothing: Whether the seasonal period accurately reflects the solar cycle
   - Both may struggle with the variable amplitude of solar cycles

3. Potential improvements:
   - Tune ARIMA parameters using techniques like AIC/BIC or grid search
   - Try different seasonal periods for exponential smoothing
   - Consider SARIMA (Seasonal ARIMA) to better capture cyclical patterns
   - Experiment with state-space models or Prophet (Facebook's forecasting tool)
   - Use ensemble methods that combine multiple forecasts

4. Limitations:
   - Both methods assume some degree of pattern stability
   - Sunspot cycles vary in length and amplitude, making precise forecasting difficult
   - Neither method incorporates physical understanding of solar dynamics
   - Long-term forecasts will degrade in accuracy beyond a few cycles
</details>

## 7. Practical Applications and Considerations

Time series analysis has numerous practical applications across domains:

### Financial Analysis
- Stock price prediction
- Volatility forecasting
- Economic indicator analysis

### Business Operations
- Sales forecasting
- Inventory management
- Capacity planning

### Environmental Science
- Climate change trend analysis
- Weather forecasting
- Natural disaster prediction

### Healthcare
- Patient monitoring
- Disease outbreak forecasting
- Hospital resource planning

## 8. Practice Exercise: Analyzing Financial Time Series

Now it's your turn to apply time series analysis to a financial dataset.

In [ ]:
# Load stock price data for a major tech company
# We'll use Yahoo Finance data through pandas_datareader
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

# Set the date range
start_date = '2015-01-01'
end_date = '2022-12-31'

# Get Apple stock data
stock_data = pdr.get_data_yahoo('AAPL', start=start_date, end=end_date)

# Focus on adjusted closing prices
stock_prices = stock_data['Adj Close']

print(f"Loaded {len(stock_prices)} trading days of Apple stock prices")
print(f"Date range: {stock_prices.index.min()} to {stock_prices.index.max()}")

# Plot the stock prices
plt.figure(figsize=(12, 6))
plt.plot(stock_prices)
plt.title('Apple Stock Price (Adjusted Close)')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(True)
plt.show()

# Your task: Analyze this stock price series by:
# 1. Computing rolling statistics (mean, standard deviation)
# 2. Detecting trends using the Theil-Sen estimator and Mann-Kendall test
# 3. Forecasting future prices using ARIMA and/or exponential smoothing
# 4. Evaluating your forecasts and discussing the results

<details>
<summary>Click for sample solution</summary>

In [ ]:
# 1. Computing rolling statistics
window_size = 20  # 20-day window (approximately one trading month)
roll_mean = rolling_mean(stock_prices, window_size)
roll_std = rolling_std(stock_prices, window_size)

plt.figure(figsize=(12, 6))
plt.plot(stock_prices, label='Apple Stock Price')
plt.plot(roll_mean, label=f'{window_size}-day Rolling Mean')
plt.plot(roll_std, label=f'{window_size}-day Rolling Std')
plt.title('Apple Stock with Rolling Statistics')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.show()

# 2. Detecting trends
# Use a 90-day window (approximately one quarter)
trend_window = 90
trends = trend_features(stock_prices, trend_window)
significant_trends = detect_trend(stock_prices, trend_window, alpha=0.05)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(stock_prices)
axes[0].set_title('Apple Stock Price')

axes[1].plot(trends['theil_sen_slope'])
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_title(f'Theil-Sen Slope ({trend_window}-day window)')

axes[2].plot(significant_trends)
axes[2].set_title('Significant Trends (Mann-Kendall p < 0.05)')

plt.tight_layout()
plt.show()

# Calculate proportion of time with significant trends
trend_count = significant_trends.sum()
trend_percentage = 100 * trend_count / len(significant_trends.dropna())
print(f"Significant trends detected in {trend_percentage:.2f}% of the windows")

# 3. Forecasting
# Split into training and test sets (last 3 months as test)
test_size = 60  # Trading days
train = stock_prices[:-test_size]
test = stock_prices[-test_size:]

# ARIMA forecasting
arima_order = (5, 1, 0)  # AR(5) with first differencing
arima_pred = arima_forecast(train, arima_order, test_size)

# Exponential smoothing
exp_pred = exponential_smoothing_forecast(
    train,
    trend='add',
    seasonal=None,  # Financial data often doesn't have clear seasonality
    steps=test_size
)

# Plot forecasts
plt.figure(figsize=(12, 6))
plt.plot(train, label='Training Data')
plt.plot(test, label='Test Data')
plt.plot(test.index, arima_pred, 'r--', label=f'ARIMA{arima_order}')
plt.plot(test.index, exp_pred, 'g--', label='Exponential Smoothing')
plt.title('Apple Stock Price Forecasts')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.show()

# 4. Evaluation
rmse_arima = sqrt(mean_squared_error(test, arima_pred))
mae_arima = mean_absolute_error(test, arima_pred)
mape_arima = 100 * np.mean(np.abs((test - arima_pred) / test))

rmse_exp = sqrt(mean_squared_error(test, exp_pred))
mae_exp = mean_absolute_error(test, exp_pred)
mape_exp = 100 * np.mean(np.abs((test - exp_pred) / test))

print("Forecast Method Comparison:")
print(f"{'Method':<20} {'RMSE':<10} {'MAE':<10} {'MAPE':<10}")
print("-" * 50)
print(f"{'ARIMA':<20} {rmse_arima:<10.2f} {mae_arima:<10.2f} {mape_arima:<10.2f}%")
print(f"{'Exponential Smoothing':<20} {rmse_exp:<10.2f} {mae_exp:<10.2f} {mape_exp:<10.2f}%")

# Discussion:
# Stock prices are notoriously difficult to predict due to their semi-random walk nature.
# Both models struggle with accurate forecasting beyond very short horizons.
# The trends detected by Theil-Sen and Mann-Kendall can help identify periods of consistent price movement.
# For investment purposes, these models would need to be supplemented with fundamental analysis,
# market sentiment indicators, and other financial factors.

</details>

## 9. Summary and Key Takeaways

In this tutorial, we've covered:

1. **Time series decomposition** - Breaking a series into trend, seasonal, and residual components
2. **Rolling statistics** - Using moving windows to analyze changing patterns
3. **Trend detection** - Applying robust methods like Theil-Sen and Mann-Kendall
4. **Change point analysis** - Identifying structural changes in time series
5. **Forecasting** - Predicting future values using ARIMA and exponential smoothing

### Next Steps

In the next tutorial, we'll explore:
- Cluster analysis and segmentation
- Unsupervised learning techniques
- Dimensionality reduction

## 10. Additional Resources

- [Time Series Analysis by Box, Jenkins, and Reinsel](https://www.wiley.com/en-us/Time+Series+Analysis%3A+Forecasting+and+Control%2C+5th+Edition-p-9781118675021)
- [Forecasting: Principles and Practice by Hyndman and Athanasopoulos](https://otexts.com/fpp3/)
- [Statsmodels Time Series Analysis Documentation](https://www.statsmodels.org/stable/tsa.html)
- [Prophet Forecasting Library](https://facebook.github.io/prophet/)
- [Time Series Analysis in Python - A Comprehensive Guide](https://www.machinelearningplus.com/time-series/time-series-analysis-python/)